In [1]:
import threading
import time
from pathlib import Path

import torch
import numpy as np
import tqdm

from diff_mfld.geodesic.geodesic_funcs import ExpMethod, LogMethod
from diff_mfld.geometry.metric import MetricField
from diff_mfld.mfld import ComputeMfld
from optim.subsolver_methods import SubsolverMethod
from optim.subsolvers.rgd import RiemGradDescentCfg
from optim.subsolvers.rtr import RiemTrustRegionCfg
from problem import create_problem

In [2]:
methods = [
    ((ExpMethod.IVP, LogMethod.BVP), "ivpbvp"),
    ((ExpMethod.APPROX_O1, LogMethod.APPROX_O1), "o1"),
    ((ExpMethod.APPROX_O2, LogMethod.APPROX_O2), "o2"),
    ((ExpMethod.APPROX_O3, LogMethod.APPROX_O3), "o3"),
]

# linear scaling of the domain
max_domain_scaling = [
    5., 10. # 0.5
]

# parameters for random starting positions
p0_mean = np.array([2., -5., -8.])
p0_covar = 2. ** 2 * np.eye(3)
num_p0 = 20

# cost minimized at this point
cost_center = np.array([-5., -1., 3.])


# the various metrics (describing the given curvature of the surface of optimization)

def euclid_metric(x1, x2, x3, scaling: float):
    metric = torch.eye(3)
    return metric


def scaled_metric(x1, x2, x3, scaling: float):
    # note that the factor is applied twice below to account for the positional scaling
    # adn then the scaling of the tangent space itself
    factor = 1. / scaling ** 2

    # elements are assigned to this metric directly to preserve gradient history
    metric = torch.zeros((3, 3))
    metric[0, 0] = factor * x1 ** 2 + 1.  # constant prevents degeneracy at origin
    metric[1, 1] = factor * x2 ** 2 + 1.
    metric[2, 2] = factor * x3 ** 2 + 1.

    metric = factor * metric  # avoid in-place operation that would destroy gradient
    return metric


def coupled_metric(x1, x2, x3, scaling: float):
    factor = 1. / scaling ** 2

    metric = torch.zeros((3, 3))
    metric[0, 0] = factor * x1 ** 2 + 1.
    metric[0, 1] = 0.5 * factor * x1 * x2
    metric[0, 3] = 0.5 * factor * x1 * x3

    metric[1, 0] = 0.5 * factor * x2 * x1
    metric[1, 1] = factor * x2 ** 2 + 1.
    metric[1, 2] = 0.5 * factor * x2 * x3

    metric[2, 0] = 0.5 * factor * x3 * x1
    metric[2, 1] = 0.5 * factor * x3 * x2
    metric[2, 2] = factor * x3 ** 2 + 1.

    metric = factor * metric
    return metric


def asymmetric_metric(x1, x2, x3, scaling: float):
    factor = 1. / scaling ** 2

    metric = torch.zeros((3, 3))
    metric[0, 0] = factor * x1 ** 2 + 1.
    metric[1, 1] = 0.5 * factor ** 2 * x1 ** 2 * x2 ** 2 + 1.
    metric[2, 2] = 0.25 * factor ** 3 * x1 ** 2 * x2 ** 2 * x3 ** 2 + 1.

    metric = factor * metric
    return metric


metric_funcs = [
    (euclid_metric, "euclid"),
    (scaled_metric, "scaled"),
    (coupled_metric, "coupled"),
    # (asymmetric_metric, "asymmetric"),
]


In [3]:
# generates the random points and scales them

# "To Everything? To the great Question of Life, the Universe and everything?"
r = np.random.default_rng(42)
p_rand_starting = r.multivariate_normal(p0_mean, p0_covar, num_p0)
max_starting_norm = np.max(np.linalg.vector_norm(p_rand_starting, axis=1))

# scales the optimization functions and starting points
p_rand_starting /= max_starting_norm
cost_center /= max_starting_norm

In [4]:
# configure the (unconstrained) configurations
optimizers = [
    ((SubsolverMethod.RIEM_GRAD_DESCENT, RiemGradDescentCfg()), "rgd"),
    ((SubsolverMethod.RIEM_TRUST_REGION, RiemTrustRegionCfg()), "rtr")
]



In [5]:
# root directory to store output files inside
base_data_dir = Path("../data/unconstrained")

In [6]:
from diff_mfld.mfld import Mfld
import itertools

# run the optimization scheme

cfgs = itertools.product(
    metric_funcs, max_domain_scaling, optimizers, methods
)
cfg_pbar = tqdm.tqdm(cfgs, desc="Configurations")

for (
        (metric_fn, metric_fn_label),
        scaling,
        ((optimizer, optimizer_cfg), optimizer_label),
        ((exp_method, log_method), method_label)
) in cfg_pbar:

    # setup the manifold configuration
    metric = MetricField(lambda x1, x2, x3: metric_fn(x1, x2, x3, scaling=scaling))
    conn = metric.christoffels()  # Levi-Civita connection (metric-compatible and torsion-free)
    mfld = Mfld(metric, conn)

    compute_mfld = ComputeMfld(
        mfld=mfld,
        exp_method=exp_method,
        log_method=log_method,
        dist_method=log_method,  # which logarithm to use when computing distance
    )

    # scale the problem coordinates accordingly
    trials_p_starting = p_rand_starting * scaling
    trials_cost_center = cost_center * scaling

    # print(f"trials_cost_center: {trials_cost_center}")
    # assert False

    # create the problem
    cost, _ = create_problem(torch.tensor(trials_cost_center))

    trials_pbar = tqdm.tqdm(range(trials_p_starting.shape[0]), desc="Trials")
    for start_idx in trials_pbar:
        p0 = torch.tensor(trials_p_starting[start_idx])
        result = optimizer(cost, p0, compute_mfld, optimizer_cfg)
        print(result)

        # save the resulting dataclass
        filename = f"{optimizer_label}/metric_{metric_fn_label}__scaling_{scaling}__trial_{start_idx}__geod_method_{method_label}.pt"
        file_path = base_data_dir / filename
        print(f"\tSaving to {file_path}")

        torch.save(result, file_path)

Configurations: 0it [00:00, ?it/s]
Trials:   5%|▌         | 1/20 [00:01<00:36,  1.89s/it]

RiemGradDescentResult(success=True, p=tensor([-1.6942, -0.3832,  0.9877]), iters=8, history=RiemGradDescentHistory(p_hist=tensor([[-0.1510, -1.6162, -0.9387],
        [-0.7861, -1.1088, -0.1459],
        [-1.1671, -0.8044,  0.3297],
        [-1.3957, -0.6217,  0.6151],
        [-1.5329, -0.5121,  0.7863],
        [-1.6152, -0.4464,  0.8891],
        [-1.6645, -0.4069,  0.9507],
        [-1.6942, -0.3832,  0.9877]]), f_hist=tensor([4.0286e+00, 1.4503e+00, 5.2210e-01, 1.8796e-01, 6.7665e-02, 2.4359e-02,
        8.7694e-03, 3.1570e-03])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_5.0__trial_0__geod_method_ivpbvp.pt



Trials:  10%|█         | 2/20 [00:03<00:36,  2.01s/it]

RiemGradDescentResult(success=True, p=tensor([-1.7075, -0.3754,  0.9955]), iters=9, history=RiemGradDescentHistory(p_hist=tensor([[ 0.1143, -1.9964, -1.7952],
        [-0.6269, -1.3369, -0.6598],
        [-1.0716, -0.9412,  0.0214],
        [-1.3384, -0.7038,  0.4301],
        [-1.4985, -0.5614,  0.6753],
        [-1.5945, -0.4759,  0.8225],
        [-1.6522, -0.4246,  0.9107],
        [-1.6867, -0.3939,  0.9637],
        [-1.7075, -0.3754,  0.9955]]), f_hist=tensor([7.1037e+00, 2.5573e+00, 9.2064e-01, 3.3143e-01, 1.1932e-01, 4.2953e-02,
        1.5463e-02, 5.5668e-03, 2.0040e-03])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_5.0__trial_1__geod_method_ivpbvp.pt



Trials:  15%|█▌        | 3/20 [00:06<00:34,  2.02s/it]

...

RiemGradDescentResult(success=True, p=tensor([-3.4187, -0.7225,  2.0133]), iters=9, history=RiemGradDescentHistory(p_hist=tensor([[ 0.0102, -2.3081, -2.2623],
        [-1.3848, -1.6631, -0.5229],
        [-2.2218, -1.2760,  0.5208],
        [-2.7239, -1.0438,  1.1470],
        [-3.0253, -0.9045,  1.5228],
        [-3.2061, -0.8208,  1.7482],
        [-3.3145, -0.7707,  1.8835],
        [-3.3796, -0.7406,  1.9646],
        [-3.4187, -0.7225,  2.0133]]), f_hist=tensor([1.6837e+01, 6.0613e+00, 2.1821e+00, 7.8555e-01, 2.8280e-01, 1.0181e-01,
        3.6650e-02, 1.3194e-02, 4.7499e-03])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_10.0__trial_16__geod_method_ivpbvp.pt



Trials:  90%|█████████ | 18/20 [00:40<00:04,  2.29s/it]

RiemGradDescentResult(success=True, p=tensor([-3.4425, -0.7245,  2.0374]), iters=10, history=RiemGradDescentHistory(p_hist=tensor([[-0.0295, -3.5806, -2.7704],
        [-1.4086, -2.4265, -0.8277],
        [-2.2361, -1.7341,  0.3379],
        [-2.7325, -1.3186,  1.0373],
        [-3.0304, -1.0694,  1.4569],
        [-3.2091, -0.9198,  1.7087],
        [-3.3164, -0.8301,  1.8597],
        [-3.3807, -0.7762,  1.9504],
        [-3.4193, -0.7439,  2.0048],
        [-3.4425, -0.7245,  2.0374]]), f_hist=tensor([2.1899e+01, 7.8837e+00, 2.8381e+00, 1.0217e+00, 3.6782e-01, 1.3242e-01,
        4.7670e-02, 1.7161e-02, 6.1780e-03, 2.2241e-03])))
	Saving to ../data/unconstrained/rgd/metric_euclid__scaling_10.0__trial_17__geod_method_ivpbvp.pt


Trials:  90%|█████████ | 18/20 [00:41<00:04,  2.32s/it]
Configurations: 8it [02:16, 17.07s/it]


KeyboardInterrupt: 